In [1]:
import json
from tqdm import tqdm
import random
from collections import Counter

In [10]:
def redistribute_answers_with_shuffle(qa_pairs):
    """
    重新分配答案并打乱选项顺序，确保每条数据的QA对答案不重复且分布均匀
    
    Args:
        qa_pairs (list): QA对列表（通常是3个）
    
    Returns:
        list: 更新后的qa_pairs
    """
    if not qa_pairs:
        return qa_pairs
    
    # 获取所有可能的选项键
    all_options = list(qa_pairs[0]["options"].keys())
    num_questions = len(qa_pairs)
    
    # 为这批QA对随机分配不重复的答案
    if num_questions <= len(all_options):
        # 如果问题数量不超过选项数量，直接随机选择不重复的答案
        selected_answers = random.sample(all_options, num_questions)
    else:
        # 如果问题数量超过选项数量，尽量均匀分布
        selected_answers = []
        full_rounds = num_questions // len(all_options)
        remaining = num_questions % len(all_options)
        
        # 先添加完整轮次
        for _ in range(full_rounds):
            selected_answers.extend(random.sample(all_options, len(all_options)))
        
        # 添加剩余的
        if remaining > 0:
            selected_answers.extend(random.sample(all_options, remaining))
        
        random.shuffle(selected_answers)
    
    # 为每个问题分配新答案并打乱选项
    for idx, qa in enumerate(qa_pairs):
        old_answer = qa["correct_answer"]
        target_answer = selected_answers[idx]
        
        # 获取当前选项内容
        options_dict = qa["options"]
        
        # 找到正确答案对应的内容
        correct_content = options_dict[old_answer]
        
        # 创建选项列表（不包含正确答案）
        other_options = [(k, v) for k, v in options_dict.items() if k != old_answer]
        
        # 打乱其他选项
        random.shuffle(other_options)
        
        # 重建选项字典，将正确答案放在目标位置
        new_options = {}
        other_idx = 0
        
        for option_key in all_options:
            if option_key == target_answer:
                # 这个位置放正确答案
                new_options[option_key] = correct_content
            else:
                # 这个位置放其他选项
                if other_idx < len(other_options):
                    new_options[option_key] = other_options[other_idx][1]
                    other_idx += 1
        
        # 更新问题
        qa["options"] = new_options
        qa["correct_answer"] = target_answer
    
    return qa_pairs


def process_qa_json(input_file, output_file):
    """
    处理整个QA JSON文件，重新分配答案分布
    
    Args:
        input_file (str): 输入文件路径
        output_file (str): 输出文件路径
    """
    # 读取数据
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # 统计处理前的答案分布
    total_before = Counter()
    total_after = Counter()
    
    # 统计每条数据内部的答案分布
    distribution_stats_before = {"all_same": 0, "two_same": 0, "all_different": 0}
    distribution_stats_after = {"all_same": 0, "two_same": 0, "all_different": 0}
    
    # 处理每个条目
    for item in tqdm(data, desc="处理QA对"):
        if "qa_pairs" in item and item["qa_pairs"]:
            # 统计处理前的分布
            answers_before = [qa["correct_answer"] for qa in item["qa_pairs"]]
            for ans in answers_before:
                total_before[ans] += 1
            
            # 统计每条数据内部答案分布情况（处理前）
            unique_before = len(set(answers_before))
            if unique_before == 1:
                distribution_stats_before["all_same"] += 1
            elif unique_before == len(answers_before):
                distribution_stats_before["all_different"] += 1
            else:
                distribution_stats_before["two_same"] += 1
            
            # 重新分配答案
            item["qa_pairs"] = redistribute_answers_with_shuffle(item["qa_pairs"])
            
            # 统计处理后的分布
            answers_after = [qa["correct_answer"] for qa in item["qa_pairs"]]
            for ans in answers_after:
                total_after[ans] += 1
            
            # 统计每条数据内部答案分布情况（处理后）
            unique_after = len(set(answers_after))
            if unique_after == 1:
                distribution_stats_after["all_same"] += 1
            elif unique_after == len(answers_after):
                distribution_stats_after["all_different"] += 1
            else:
                distribution_stats_after["two_same"] += 1
    
    # 保存结果
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    # 打印统计信息
    print("\n" + "="*50)
    print("整体答案分布:")
    print("\n处理前:")
    for option in sorted(total_before.keys()):
        print(f"  {option}: {total_before[option]} ({total_before[option]/sum(total_before.values())*100:.1f}%)")
    
    print("\n处理后:")
    for option in sorted(total_after.keys()):
        print(f"  {option}: {total_after[option]} ({total_after[option]/sum(total_after.values())*100:.1f}%)")
    
    print("\n" + "="*50)
    print("每条数据内部答案分布情况:")
    print("\n处理前:")
    total_items = sum(distribution_stats_before.values())
    print(f"  3个答案都相同: {distribution_stats_before['all_same']} ({distribution_stats_before['all_same']/total_items*100:.1f}%)")
    print(f"  有2个答案相同: {distribution_stats_before['two_same']} ({distribution_stats_before['two_same']/total_items*100:.1f}%)")
    print(f"  3个答案都不同: {distribution_stats_before['all_different']} ({distribution_stats_before['all_different']/total_items*100:.1f}%)")
    
    print("\n处理后:")
    print(f"  3个答案都相同: {distribution_stats_after['all_same']} ({distribution_stats_after['all_same']/total_items*100:.1f}%)")
    print(f"  有2个答案相同: {distribution_stats_after['two_same']} ({distribution_stats_after['two_same']/total_items*100:.1f}%)")
    print(f"  3个答案都不同: {distribution_stats_after['all_different']} ({distribution_stats_after['all_different']/total_items*100:.1f}%)")
    print("="*50)


# 使用示例
process_qa_json("../fox_data/qa/qa.json", "../fox_data/qa/qa_redistributed.json")

处理QA对: 100%|██████████| 112/112 [00:00<00:00, 18726.81it/s]


整体答案分布:

处理前:
  A: 72 (21.4%)
  B: 158 (47.0%)
  C: 90 (26.8%)
  D: 16 (4.8%)

处理后:
  A: 94 (28.0%)
  B: 77 (22.9%)
  C: 82 (24.4%)
  D: 83 (24.7%)

每条数据内部答案分布情况:

处理前:
  3个答案都相同: 29 (25.9%)
  有2个答案相同: 62 (55.4%)
  3个答案都不同: 21 (18.8%)

处理后:
  3个答案都相同: 0 (0.0%)
  有2个答案相同: 0 (0.0%)
  3个答案都不同: 112 (100.0%)


In [ ]:
with open("../fox_data/qa/qa.json", "r", encoding="utf-8") as f:
    data = json.load(f)

a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds


## 检查qa对

In [2]:
import json

def check_model_consistency(qwen_file, gpt_file, deepseek_file):
    """
    检查三个模型的检查结果文件，找出其中三个模型都判断一致的数据，和剩下的其他数据。
    
    参数:
    qwen_file (str): qwen3_check.json 文件路径
    gpt_file (str): gptoss.json 文件路径
    deepseek_file (str): dpskv3_check.json 文件路径
    
    返回:
    tuple: (consistent_items, inconsistent_items)
        consistent_items: 三个模型判断一致的item列表（每个item是完整的字典）
        inconsistent_items: 不一致的item列表
    """
    # 加载三个文件
    with open(qwen_file, 'r', encoding='utf-8') as f:
        qwen_data = json.load(f)
    with open(gpt_file, 'r', encoding='utf-8') as f:
        gpt_data = json.load(f)
    with open(deepseek_file, 'r', encoding='utf-8') as f:
        deepseek_data = json.load(f)
    
    # 假设三个文件的数据顺序相同（按image排序）
    consistent_items = []
    inconsistent_items = []
    
    for idx, (qwen_item, gpt_item, deepseek_item) in enumerate(zip(qwen_data, gpt_data, deepseek_data)):
        # 检查image是否匹配（确保是同一个item）
        if qwen_item['image'] != gpt_item['image'] or qwen_item['image'] != deepseek_item['image']:
            raise ValueError(f"Item {idx} images do not match: {qwen_item['image']} vs {gpt_item['image']} vs {deepseek_item['image']}")
        
        qwen_checks = qwen_item.get('check_results', [])
        gpt_checks = gpt_item.get('check_results', [])
        deepseek_checks = deepseek_item.get('check_results', [])
        
        # 检查长度是否相同
        if len(qwen_checks) != len(gpt_checks) or len(qwen_checks) != len(deepseek_checks):
            inconsistent_items.append(qwen_item)
            continue
        
        is_consistent = True
        for i, (q_check, g_check, d_check) in enumerate(zip(qwen_checks, gpt_checks, deepseek_checks)):
            # 检查verdict
            if q_check['verdict'] != g_check['verdict'] or q_check['verdict'] != d_check['verdict']:
                is_consistent = False
                break
            # 如果都是"No"，检查correct_answer_should_be
            if q_check['verdict'] == 'No':
                if (q_check.get('correct_answer_should_be') != g_check.get('correct_answer_should_be') or
                    q_check.get('correct_answer_should_be') != d_check.get('correct_answer_should_be')):
                    is_consistent = False
                    break
        
        if is_consistent:
            consistent_items.append(qwen_item)
        else:
            inconsistent_items.append(qwen_item)
    
    return consistent_items, inconsistent_items

In [ ]:

# # 示例使用（假设文件路径）
# # 在调用check_model_consistency后，添加以下代码来打印不一致数据的image

# consistent, inconsistent = check_model_consistency(
#     "../fox_data/qa/check/qwen3_check.json",
#     "../fox_data/qa/check/gptoss_check.json",
#     "../fox_data/qa/check/dpskv3_check.json"
# )

# print(f"Consistent items: {len(consistent)}")
# print(f"Inconsistent items: {len(inconsistent)}")

# # 打印不一致数据的image
# print("\nInconsistent items' images:")
# for item in inconsistent:
#     print(item['image'])

In [10]:
# 在调用check_model_consistency后，添加以下代码来详细展示不一致数据的差异

consistent, inconsistent = check_model_consistency(
    "../fox_data/qa/check/qwen3_check.json",
    "../fox_data/qa/check/gptoss_check.json",
    "../fox_data/qa/check/dpskv3_check.json"
)

print(f"Consistent items: {len(consistent)}")
print(f"Inconsistent items: {len(inconsistent)}")

# 为了详细展示，需要重新加载数据来比较
with open("../fox_data/qa/check/qwen3_check.json", 'r', encoding='utf-8') as f:
    qwen_data = json.load(f)
with open("../fox_data/qa/check/gptoss_check.json", 'r', encoding='utf-8') as f:
    gpt_data = json.load(f)
with open("../fox_data/qa/check/dpskv3_check.json", 'r', encoding='utf-8') as f:
    deepseek_data = json.load(f)

Consistent items: 94
Inconsistent items: 18


In [11]:
# 打印不一致数据的详细信息
print("\nDetailed inconsistent items:")
for item in inconsistent:
    image = item['image']
    print(f"\nImage: {image}")
    
    # 找到对应的item在其他数据中的索引（假设顺序相同）
    idx = next(i for i, it in enumerate(qwen_data) if it['image'] == image)
    qwen_checks = qwen_data[idx].get('check_results') or []
    gpt_checks = gpt_data[idx].get('check_results') or []
    deepseek_checks = deepseek_data[idx].get('check_results') or []
    
    for i in range(len(qwen_checks)):
        q_verdict = qwen_checks[i]['verdict']
        g_verdict = gpt_checks[i]['verdict']
        d_verdict = deepseek_checks[i]['verdict']
        
        is_same_verdict = q_verdict == g_verdict == d_verdict
        is_same_correct = True
        if q_verdict == 'No':
            q_correct = qwen_checks[i].get('correct_answer_should_be')
            g_correct = gpt_checks[i].get('correct_answer_should_be')
            d_correct = deepseek_checks[i].get('correct_answer_should_be')
            is_same_correct = q_correct == g_correct == d_correct
        
        if not (is_same_verdict and is_same_correct):
            print(f"  Question {i+1}:")
            print(f"    Qwen: {q_verdict}")
            print(f"    GPT: {g_verdict}")
            print(f"    DeepSeek: {d_verdict}")
            
            if q_verdict == 'No':
                print(f"      Qwen correct_should_be: {q_correct}")
            if g_verdict == 'No':
                print(f"      GPT correct_should_be: {g_correct}")
            if d_verdict == 'No':
                print(f"      DeepSeek correct_should_be: {d_correct}")
            
            # 打印原始QA对
            original_qa = item['qa_pairs'][i]
            print(f"    Original QA:")
            for key, value in original_qa.items():
                print(f"      {key}: {value}")


Detailed inconsistent items:

Image: en_3.png
  Question 2:
    Qwen: Yes
    GPT: No
    DeepSeek: Yes


NameError: name 'g_correct' is not defined

## 再次复查核对后的数据

In [5]:
consistent, inconsistent = check_model_consistency(
    "../fox_data/qa/check/qwen3_recheck.json",
    "../fox_data/qa/check/gptoss_recheck.json",
    "../fox_data/qa/check/dpskv3_recheck.json"
)

print(f"Consistent items: {len(consistent)}")
print(f"Inconsistent items: {len(inconsistent)}")

# 为了详细展示，需要重新加载数据来比较
with open("../fox_data/qa/check/qwen3_recheck.json", 'r', encoding='utf-8') as f:
    qwen_data = json.load(f)
with open("../fox_data/qa/check/gptoss_recheck.json", 'r', encoding='utf-8') as f:
    gpt_data = json.load(f)
with open("../fox_data/qa/check/dpskv3_recheck.json", 'r', encoding='utf-8') as f:
    deepseek_data = json.load(f)

Consistent items: 105
Inconsistent items: 7


In [7]:
# 打印不一致数据的详细信息
print("\nDetailed inconsistent items:")
for item in inconsistent:
    image = item['image']
    print(f"\nImage: {image}")
    
    # 找到对应的item在其他数据中的索引（假设顺序相同）
    idx = next(i for i, it in enumerate(qwen_data) if it['image'] == image)
    qwen_checks = qwen_data[idx].get('check_results') or []
    gpt_checks = gpt_data[idx].get('check_results') or []
    deepseek_checks = deepseek_data[idx].get('check_results') or []
    
    for i in range(len(qwen_checks)):
        q_verdict = qwen_checks[i]['verdict']
        g_verdict = gpt_checks[i]['verdict']
        d_verdict = deepseek_checks[i]['verdict']
        
        is_same_verdict = q_verdict == g_verdict == d_verdict
        is_same_correct = True
        if q_verdict == 'No':
            q_correct = qwen_checks[i]['correct_answer_should_be']
            g_correct = gpt_checks[i]['correct_answer_should_be']
            d_correct = deepseek_checks[i]['correct_answer_should_be']
            is_same_correct = q_correct == g_correct == d_correct
        
        if not (is_same_verdict and is_same_correct):
            print(f"  Question {i+1}:")
            print(f"    Qwen: {q_verdict}")
            print(f"    GPT: {g_verdict}")
            print(f"    DeepSeek: {d_verdict}")
            
            # if q_verdict == 'No':
            #     print(f"      Qwen correct_should_be: {q_correct}")
            # if g_verdict == 'No':
            #     print(f"      GPT correct_should_be: {g_correct}")
            # if d_verdict == 'No':
            #     print(f"      DeepSeek correct_should_be: {d_correct}")
            
            # 打印原始QA对
            original_qa = item['qa_pairs'][i]
            print(f"    Original QA:")
            for key, value in original_qa.items():
                print(f"      {key}: {value}")


Detailed inconsistent items:

Image: en_3.png
  Question 2:
    Qwen: Yes
    GPT: No
    DeepSeek: Yes
    Original QA:
      question: What physical traits did Half-a-chick lack at the start of his journey?
      options: {'A': 'A beak and claws', 'B': 'A tail and feathers', 'C': 'One leg, one eye, and one wing', 'D': 'Two legs, two eyes, and two wings'}
      correct_answer: C
      explanation: The text describes Half-a-chick as having 'one leg, one eye, and one wing' when he decides to travel to the capital.

Image: en_14.png
  Question 2:
    Qwen: Yes
    GPT: No
    DeepSeek: Yes
    Original QA:
      question: Under Section IV, what is the latest applicability date for state/local government entities requiring legislative action to comply with market reforms?
      options: {'A': 'December 31, 2013', 'B': 'September 13, 2014', 'C': 'First day of the first plan year following the first legislative session after September 13, 2013', 'D': 'January 1, 2014'}
      correct_answer